# Phase 1.5 1a — A.1 / A.2 / A.3 strict-control ladder

Paper §7.4 Gap A control ladder — the Q-only bottleneck attribution test:

- **A.1** (default): Q-only encoding.
- **A.2** (strict control): Q + position-masked-P. Positions preserved, content erased.
- **A.3** (ceiling): Q + full-P-cross-encoded. Bottleneck violation upper bound.

**Attribution rule**: A.1 must ≥ A.2 (Q-only beats partial-Q-leak control); A.1 should
trail A.3 by less than the topic-axis trail (bottleneck pays its cost on operation, not
topic). Runtime ~6h Colab T4 / 1.5h A100.

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path

BASE = Path('/content/drive/MyDrive/sideproject')
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'BASE = {BASE}, device = {device}')

## 2. Run A.1 + A.2 + A.3 (sequential)

In [ ]:
from phase1_5.ablations import PHASE1_5_INITIAL_ROWS, run_all_rows
from phase1_5.data import MCCorpusConfig
from phase1_5.train import TrainConfig

corpus_cfg = MCCorpusConfig(
    max_train_samples=20_000,
    max_val_samples=2_000,
    max_test_samples=2_000,
    t_cap_q=128,
    t_cap_p=256,
    cache_root=str(BASE / 'out' / 'phase1_5' / 'cache'),
)
train_cfg = TrainConfig(epochs=40, lr=1e-3, k_target=4.0, seed=0)

df = run_all_rows(
    PHASE1_5_INITIAL_ROWS,
    corpus_cfg=corpus_cfg,
    train_cfg=train_cfg,
    out_dir=str(BASE / 'out' / 'phase1_5' / 'ablations'),
    device=device,
    seed=0,
    batch_size=64,
    skip_if_exists=True,
)
df

## 3. Comparison table

In [ ]:
cols = ['verdict', 'adj_operation', 'sigma_adj_operation', 'threshold',
        'adj_random', 'adj_topic', 'adj_token', 'adj_geometry',
        'k_active_mean', 'dead_expert_frac', 'val_mc_acc_final']
df[cols].style.format({c: '{:.4f}' for c in cols if c not in ('verdict',)})

## 4. Bar chart — adj_operation vs row with 1σ error + threshold line

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rows = df.index.tolist()
vals = df['adj_operation'].to_numpy(dtype=float)
errs = df['sigma_adj_operation'].to_numpy(dtype=float)
thr = df['threshold'].to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(rows))
colors = ['tab:blue' if v else 'tab:red' for v in df['passes_sigma_gate']]
ax.bar(x, vals, yerr=errs, capsize=4, color=colors)
for xi, t in zip(x, thr):
    ax.hlines(t, xi - 0.4, xi + 0.4, colors='gray', linestyles='--', label='threshold' if xi == 0 else None)
ax.set_xticks(x); ax.set_xticklabels(rows)
ax.set_ylabel('adj_operation')
ax.set_title('1σ-bootstrap gate: adj_operation vs row')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout(); plt.show()

## 5. Ladder verification (A.1 ≥ A.2; A.1 trail to A.3 < topic-axis trail)

In [ ]:
adj = df['adj_operation']
a1, a2, a3 = adj.loc['A.1'], adj.loc['A.2'], adj.loc['A.3']
topic_trail = df.loc['A.1', 'adj_topic'] - df.loc['A.3', 'adj_topic']
op_trail = a3 - a1

print(f'A.1 adj_op  = {a1:.4f}')
print(f'A.2 adj_op  = {a2:.4f}')
print(f'A.3 adj_op  = {a3:.4f}')
print(f'\nA.1 ≥ A.2  : {bool(a1 >= a2)}')
print(f'A.1 trail to A.3 on operation = {op_trail:.4f}')
print(f'A.1 trail to A.3 on topic     = {topic_trail:.4f}')
print(f'\noperation trail < topic trail (bottleneck attribution): {bool(op_trail < topic_trail)}')

## 6. Save summary table

In [ ]:
out_csv = BASE / 'out' / 'phase1_5' / 'ablations' / 'initial_3row.csv'
out_csv.parent.mkdir(parents=True, exist_ok=True)
df[cols].to_csv(out_csv)
print(f'saved → {out_csv}')